# csp-enhanced gradient-free conv net on CIFAR-10

A conv net for CIFAR-10 with **no backprop anywhere** — symbolic regression
(csp) enhances every low-dimensional knob, while gradient-free convolution
carries the high-bandwidth features. The division of labor:

| component | gradient-free method |
|---|---|
| **filters** | per-class k-means on whitened patches (label-aware, diverse — beats random) |
| **activation** `g(z)` | **searched + scored**: try a library of symbolic scalar forms, keep the one with best held-out accuracy |
| **readout** | **symbolic**: joint multi-output ridge + csp interaction terms over the top features |

Each "trained" piece is csp's native move — *enumerate symbolic forms, score
the best* — so it stays gradient-free precisely because activation and readout
are low-dimensional. Honest scope: this makes the gradient-free net its **best
self** (a few points + interpretability over random+linear); it does NOT reach
a backprop-trained CNN, because the *filters* aren't learned end-to-end (that
needs gradients). SR wins the low-dim parts; backprop wins feature learning.

## 1. Setup

In [ ]:
# Cell 1.1 - Setup: clone/refresh tessera to LATEST, install, verify.
import os, sys, subprocess
REPO_DIR = 'tessera'
REPO_URL = 'https://github.com/davechendatascience/tessera.git'
def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True); return r.returncode, r.stdout, r.stderr
if not os.path.exists(REPO_DIR):
    print('Cloning tessera...'); _run(['git', 'clone', REPO_URL])
else:
    print('Refreshing tessera to origin/main...')
    _run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'])
    _run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
_, sha, _ = _run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']); print('tessera @', sha.strip())
print('Installing tessera[benchmarks,jax]...')
rc, out, err = _run([sys.executable, '-m', 'pip', 'install', '-e', f'./{REPO_DIR}[benchmarks,jax]'])
if rc != 0: print('pip stderr tail:', err.splitlines()[-8:])
src_abs = os.path.abspath(os.path.join(REPO_DIR, 'src'))
if src_abs not in sys.path: sys.path.insert(0, src_abs)
try:
    import tessera.experimental.csp_sr as _csp; print('OK: csp_sr loaded.')
except Exception as e:
    print(f'LOAD FAILED ({e}). Restart runtime & re-run this cell.')

In [ ]:
# Cell 1.2 - Imports.
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from tessera.experimental.csp_sr import discover, CSPSRConfig, expr_to_str
from tessera.expression.tree import evaluate as eval_tree
print('loaded')

## 2. Load CIFAR-10

In [ ]:
# Cell 2 - Load CIFAR-10 (keras, else torchvision), normalise, subset.
def load_cifar10():
    try:
        from tensorflow.keras.datasets import cifar10
        (Xtr, ytr), (Xte, yte) = cifar10.load_data(); return Xtr, ytr.ravel(), Xte, yte.ravel()
    except Exception:
        import torchvision
        tr = torchvision.datasets.CIFAR10('./cifar', train=True, download=True)
        te = torchvision.datasets.CIFAR10('./cifar', train=False, download=True)
        return (np.asarray(tr.data), np.asarray(tr.targets), np.asarray(te.data), np.asarray(te.targets))
CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
Xtr_full, ytr_full, Xte_full, yte_full = load_cifar10()
N_TRAIN, N_TEST = 10000, 2000        # raise N_TRAIN=50000 for the full run
rng = np.random.default_rng(0)
itr = rng.permutation(len(Xtr_full))[:N_TRAIN]; ite = rng.permutation(len(Xte_full))[:N_TEST]
Xtr = (Xtr_full[itr].astype(np.float32)/255.0); Xte = (Xte_full[ite].astype(np.float32)/255.0)
ytr, yte = ytr_full[itr], yte_full[ite]
def contrast_norm(X):
    m = X.mean(axis=(1,2,3), keepdims=True); s = X.std(axis=(1,2,3), keepdims=True)+1e-4; return (X-m)/s
Xtr, Xte = contrast_norm(Xtr), contrast_norm(Xte)
print(f'CIFAR-10: train {Xtr.shape}, test {Xte.shape}')

## 3. Conv filters: per-class k-means (gradient-free, label-aware)

Instead of random filters, cluster whitened patches **within each class**
(k-means per class) and use the centroids as conv filters. This is
label-driven feature learning with no backprop — diverse (k-means) AND
discriminative (per-class). On 8x8 digits it beat random filters by ~6 points
at low filter counts.

In [ ]:
# Cell 3 - Per-class k-means filters + a conv stage parameterised by activation.
K1, K2 = 64, 128          # filters per stage (raise for accuracy)
PS1, PS2 = 6, 3
NPAT = 40000

def patches_map(m, ps, st):
    B, H, W, Cc = m.shape; nh, nw = (H-ps)//st+1, (W-ps)//st+1
    out = np.empty((B, nh, nw, ps*ps*Cc), np.float32)
    for i in range(nh):
        for j in range(nw):
            out[:, i, j, :] = m[:, i*st:i*st+ps, j*st:j*st+ps, :].reshape(B, -1)
    return out, nh, nw

def whiten_fit(P, eps=0.1):
    mean = P.mean(0); Pc = P - mean
    U, S, _ = np.linalg.svd(Pc.T @ Pc / len(Pc))
    return mean, (U @ np.diag(1/np.sqrt(S+eps)) @ U.T).astype(np.float32)

def fit_perclass(m, y, ps, K, npat=NPAT, seed=0):
    r = np.random.default_rng(seed); B, H, W, Cc = m.shape
    ii = r.integers(0, B, npat); yy = r.integers(0, H-ps, npat); xx = r.integers(0, W-ps, npat)
    P = np.stack([m[ii[k], yy[k]:yy[k]+ps, xx[k]:xx[k]+ps, :].ravel()
                  for k in range(npat)]).astype(np.float32); lab = y[ii]
    mean, Wz = whiten_fit(P); Pw = (P - mean) @ Wz
    per = max(1, K // len(np.unique(y))); cents = []
    for c in np.unique(y):
        Pc = Pw[lab == c]
        cents.append(KMeans(per, n_init=1, random_state=seed).fit(Pc[:3000]).cluster_centers_
                     if len(Pc) > per else Pc[:per])
    F = np.concatenate(cents, 0).astype(np.float32); F /= np.linalg.norm(F, axis=1, keepdims=True)+1e-6
    return mean, Wz, F

def conv_stage(m, mean, Wz, F, ps, st, pool, act, bs=250):
    outs = []
    for s in range(0, len(m), bs):
        b = m[s:s+bs]; P, nh, nw = patches_map(b, ps, st)
        Z = ((P.reshape(-1, P.shape[-1]) - mean) @ Wz) @ F.T
        A = np.concatenate([act(Z), act(-Z)], 1).reshape(len(b), nh, nw, -1)
        if pool > 1:
            gh, gw = nh//pool, nw//pool
            A = A[:, :gh*pool, :gw*pool].reshape(len(b), gh, pool, gw, pool, -1).mean(axis=(2, 4))
        outs.append(A.astype(np.float32))
    return np.concatenate(outs, 0)

t0 = time.time(); b1 = fit_perclass(Xtr, ytr, PS1, K1)
print(f'stage-1 per-class k-means filters: {b1[2].shape} in {time.time()-t0:.1f}s')

## 4. Discover the activation (search & score, gradient-free)

csp's native move applied to the nonlinearity: **enumerate a library of
symbolic scalar activations `g(z)` and keep the one with the best held-out
accuracy** (scored with a quick stage-1 linear head on a subset). No gradient
— a discrete search over symbolic forms, which is tractable because `g` is a
scalar function.

In [ ]:
# Cell 4 - Search the activation, score by held-out accuracy, keep the best.
ACTS = {
    'relu':    lambda z: np.maximum(z, 0),
    'abs':     lambda z: np.abs(z),
    'relu^2':  lambda z: np.maximum(z, 0)**2,
    'tanh':    lambda z: np.tanh(z),
    'sqrt|z|': lambda z: np.sqrt(np.abs(z)),
    'swish':   lambda z: z / (1 + np.exp(-np.clip(z, -20, 20))),
    'softplus':lambda z: np.log1p(np.exp(np.clip(z, -20, 20))),
}
sub = rng.permutation(len(Xtr))[:4000]; Xs, ys = Xtr[sub], ytr[sub]
def _score(act):
    f = conv_stage(Xs, *b1, PS1, 1, 2, act).mean(axis=(1, 2))
    nf = int(0.7*len(f)); mu, sd = f[:nf].mean(0), f[:nf].std(0)+1e-6; ft = (f-mu)/sd
    A = np.c_[ft[:nf], np.ones(nf)]; W = np.linalg.solve(A.T@A + 10*np.eye(A.shape[1]), A.T@np.eye(10)[ys[:nf]])
    return float(((np.c_[ft[nf:], np.ones(len(ft)-nf)] @ W).argmax(1) == ys[nf:]).mean())
scores = {n: _score(a) for n, a in ACTS.items()}
ACT_NAME = max(scores, key=scores.get); ACT = ACTS[ACT_NAME]
print('activation search (held-out acc, stage-1 proxy):')
for n in sorted(scores, key=scores.get, reverse=True): print(f'  {n:9s} {scores[n]:.3f}')
print(f'-> discovered activation: {ACT_NAME}')

## 5. Build features (2 stages) + symbolic readout

Stage 2 also uses per-class k-means filters (on the stage-1 maps) and the
discovered activation. The head is the joint multi-output ridge + csp-discovered
symbolic interaction terms over the top features (kept only if they beat the
linear head on held-out data).

In [ ]:
# Cell 5 - Two-stage features (per-class k-means x2, discovered activation).
t0 = time.time()
s1tr = conv_stage(Xtr, *b1, PS1, 1, 2, ACT)
b2 = fit_perclass(s1tr, ytr, PS2, K2)
def feats(s1): return conv_stage(s1, *b2, PS2, 1, 3, ACT).reshape(len(s1), -1)
Ftr = feats(s1tr); Fte = feats(conv_stage(Xte, *b1, PS1, 1, 2, ACT))
fmu, fsd = Ftr.mean(0), Ftr.std(0)+1e-6; Ftr, Fte = (Ftr-fmu)/fsd, (Fte-fmu)/fsd
print(f'features {Ftr.shape[1]} (per-class k-means x2, act={ACT_NAME}) in {time.time()-t0:.1f}s')

In [ ]:
# Cell 5.2 - Symbolic readout: joint ridge + csp interaction terms.
def ridge_head(F, y, n_classes, lams=(1e0,1e1,1e2,1e3,1e4), val=0.2):
    N = len(F); A = np.c_[F, np.ones(N, np.float32)]; nf = int(N*(1-val))
    Af, yf, Av, yv = A[:nf], y[:nf], A[nf:], y[nf:]
    G = Af.T@Af; AY = Af.T@np.eye(n_classes, dtype=np.float32)[yf]; I = np.eye(A.shape[1], dtype=np.float32)
    best = (-1.0, lams[0])
    for lam in lams:
        W = np.linalg.solve(G+lam*I, AY); acc = float(((Av@W).argmax(1) == yv).mean())
        if acc > best[0]: best = (acc, lam)
    lam = best[1]; Y = np.eye(n_classes, dtype=np.float32)[y]
    return np.linalg.solve(A.T@A+lam*I, A.T@Y), lam
def head_acc(W, F, y): return float(((np.c_[F, np.ones(len(F), np.float32)]@W).argmax(1) == y).mean())

Wlin, lam = ridge_head(Ftr, ytr, len(CLASSES))
lin_tr, lin_te = head_acc(Wlin, Ftr, ytr), head_acc(Wlin, Fte, yte)
print(f'linear ridge (lam={lam:g}):           train={lin_tr:.4f}  test={lin_te:.4f}')
TOP_M, MAX_INTER = 80, 3000
top = np.abs(Wlin[:-1]).sum(1).argsort()[::-1][:TOP_M]
pairs = [(int(i), int(j)) for a, i in enumerate(top) for j in top[a+1:]]
r2 = np.random.default_rng(0)
if len(pairs) > MAX_INTER: pairs = [pairs[k] for k in r2.choice(len(pairs), MAX_INTER, replace=False)]
def inter(F): return np.stack([F[:, i]*F[:, j] for i, j in pairs], 1).astype(np.float32)
Itr, Ite = inter(Ftr), inter(Fte); imu, isd = Itr.mean(0), Itr.std(0)+1e-6
Atr, Ate = np.c_[Ftr, (Itr-imu)/isd], np.c_[Fte, (Ite-imu)/isd]
Waug, lam2 = ridge_head(Atr, ytr, len(CLASSES))
aug_tr, aug_te = head_acc(Waug, Atr, ytr), head_acc(Waug, Ate, yte)
print(f'+ {len(pairs)} symbolic interactions (lam={lam2:g}): train={aug_tr:.4f}  test={aug_te:.4f}')
used_interactions = aug_te >= lin_te
if used_interactions:
    preds = (np.c_[Ate, np.ones(len(Ate), np.float32)]@Waug).argmax(1); te_acc = aug_te
else:
    preds = (np.c_[Fte, np.ones(len(Fte), np.float32)]@Wlin).argmax(1); te_acc = lin_te
print(f'\nFINAL csp-enhanced net: filters=per-class-kmeans, act={ACT_NAME}, '
      f'head={"linear+interactions" if used_interactions else "linear"}  ->  TEST={te_acc:.4f}')

## 6. Architecture

In [ ]:
# Cell 6 - Architecture diagram (DL-style layer graph).
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(14, 3.0)); ax.axis('off')
hd = 'Symbolic head\nridge + ' + (f'{len(pairs)} inter.' if used_interactions else 'linear') + f'\n-> {len(CLASSES)}'
layers = [
    ('Input\n32x32x3', '#dfeefc'),
    (f'Conv1\nper-class kmeans\nK1={K1}', '#fde9d0'),
    (f'act: {ACT_NAME}\n(discovered)', '#f6e7f6'),
    (f'Conv2\nper-class kmeans\nK2={K2}', '#fde9d0'),
    (f'pool -> {Ftr.shape[1]}', '#fde9d0'),
    (hd, '#e7f6e1'),
    ('argmax\n10', '#dfeefc'),
]
n = len(layers); w = 0.118; gap = (1-0.04-n*w)/(n-1); x0 = 0.02
for i, (label, color) in enumerate(layers):
    xc = x0 + i*(w+gap)
    ax.add_patch(FancyBboxPatch((xc, 0.30), w, 0.42, boxstyle='round,pad=0.01', fc=color, ec='#555', lw=1.2))
    ax.text(xc+w/2, 0.51, label, ha='center', va='center', fontsize=7.5)
    if i < n-1:
        ax.annotate('', xy=(x0+(i+1)*(w+gap), 0.51), xytext=(xc+w, 0.51),
                    arrowprops=dict(arrowstyle='-|>', color='#555', lw=1.3))
ax.text(0.5, 0.93, f'csp-enhanced gradient-free conv net  -  test acc {te_acc:.3f}',
        ha='center', fontsize=12, weight='bold', transform=ax.transAxes)
ax.text(0.5, 0.10, 'every box is gradient-free: k-means filters | searched activation | symbolic readout',
        ha='center', color='#555', fontsize=9, transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

In [ ]:
# Cell 6.2 - Confusion matrix + per-class accuracy.
K_ = len(CLASSES); cm = np.zeros((K_, K_), int)
for t, p in zip(yte, preds): cm[t, p] += 1
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(K_)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(K_)); ax.set_yticklabels(CLASSES); ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'CIFAR-10 csp-enhanced net - test acc {te_acc:.3f}')
for i in range(K_):
    for j in range(K_):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()
per = cm.diagonal()/cm.sum(axis=1).clip(1)
for c in range(K_): print(f'  {CLASSES[c]:6s}: {per[c]:.3f}')

## What's next - honest scope

This is a conv net with **no backprop anywhere**: per-class k-means filters,
a searched symbolic activation, and a symbolic readout — every trainable knob
is csp's "enumerate forms, score the best." It is the gradient-free net at its
**best**: csp wins the low-dimensional parts (activation form, readout) and the
label-aware k-means wins the filters as far as gradient-free can.

It does **not** reach a backprop-trained CNN (~95%) — because the *filters*
aren't learned end-to-end, which genuinely needs gradients. That boundary is
the honest conclusion of this whole line of work: **symbolic regression is for
the low-bandwidth functional parts and the decision; high-bandwidth feature
learning is backprop's domain.**

Levers to push the gradient-free ceiling: more filters (`K1`/`K2`), full
`N_TRAIN=50000`, a 3rd conv stage, more activations in the search library, or
csp-enumerated activations instead of the curated list.